In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)

from src.data.load_data import load_bank_data
from src.data.preprocess import split_data, BankFeatureEngineer, build_enhanced_pipeline

from src.models.logistic import (
    train_logistic_with_tuning,
    find_best_threshold,
    save_logistic_model
)

# Suppressing numerical stability warnings (overflow/underflow)
# These do not affect final model performance
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [2]:
# Load data
df = load_bank_data("../data/raw/bank-full.csv")

# Split
X_train, X_test, y_train, y_test = split_data(df)

# Feature engineering detection
fe = BankFeatureEngineer()
X_tmp = fe.fit_transform(X_train)

num_cols = X_tmp.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_tmp.select_dtypes(include=["object", "category"]).columns.tolist()

# Preprocessing
preprocess_pipeline = build_enhanced_pipeline(num_cols, cat_cols)

logistic_model, best_params, best_cv_score = train_logistic_with_tuning(
    preprocess_pipeline,
    X_train,
    y_train
)

print("Best Parameters:", best_params)
print("Best CV F1 Score:", best_cv_score)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Parameters: {'classifier__C': 1, 'classifier__solver': 'liblinear'}
Best CV F1 Score: 0.3889763742863221


In [3]:
# Find best threshold
best_threshold, best_f1 = find_best_threshold(logistic_model, X_test, y_test)

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

# Probabilities
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Final predictions
y_pred = (y_prob >= best_threshold).astype(int)

print("\n--- LOGISTIC REGRESSION ---")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1       :", f1_score(y_test, y_pred, zero_division=0))
print("MCC      :", matthews_corrcoef(y_test, y_pred))

# Save
save_logistic_model(logistic_model, best_threshold)

Best threshold: 0.6224489795918368
Best F1: 0.4568345323741007

--- LOGISTIC REGRESSION ---
Accuracy : 0.8664160123852703
AUC      : 0.7772986447888467
Precision: 0.43567753001715265
Recall   : 0.48015122873345933
F1       : 0.4568345323741007
MCC      : 0.381467201567676


### Logistic Regression Observations

- The dataset is highly imbalanced (~88% 'no', ~12% 'yes'), so F1-score is used for evaluation.
- Class imbalance is handled using class_weight="balanced".
- Hyperparameter tuning using GridSearchCV selected C=1 as optimal.

- AUC score (~0.77) indicates good ranking ability of the model.
- Default threshold (0.5) is not optimal for imbalanced data.

- After threshold tuning, best threshold ≈ 0.62 improved F1-score.
- Final F1 score ≈ 0.457 shows moderate performance.

- Precision (~0.43) and Recall (~0.48) indicate a trade-off:
  → Model captures more positives (higher recall)
  → But at cost of some false positives

- MCC (~0.38) confirms balanced predictive performance.

Conclusion:
Logistic Regression provides a strong baseline but may struggle with complex non-linear patterns.
